In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset

from jlab import MLPClassifier, plot_comparison_with_pull, prepare_data, train_gan_model
from jlab import MLPVAE, train_vae_model

In [ ]:
# Set global font sizes and thickness
plt.rcParams.update(
    {
        "font.size": 16,
        "axes.titlesize": 18,
        "axes.labelsize": 16,
        "xtick.labelsize": 12,
        "ytick.labelsize": 12,
        "legend.fontsize": 14,
        "axes.linewidth": 2.0,
        "lines.linewidth": 2.5,
    }
)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\tUsing device {device}")

In [ ]:
seed = 42
np.random.seed(seed)
torch.manual_seed(seed)

## Vanilla GAN

In [ ]:
# Load and preprocess data
data_path = "./data/eICU_age.npy"
age_values, age_scaled, scaler = prepare_data(data_path)

# Split data
train_age, val_age = train_test_split(age_scaled, test_size=0.1)

# Create DataLoaders
batch_size = 64
train_loader = DataLoader(
    TensorDataset(torch.from_numpy(train_age)), batch_size=batch_size, shuffle=True
)
val_loader = DataLoader(
    TensorDataset(torch.from_numpy(val_age)), batch_size=batch_size, shuffle=False
)

In [ ]:
# Hyperparameters
latent_size = 16
learning_rate = 0.001
num_epochs = 100
n_samples = 100000

In [ ]:
# Models
model_G = MLPClassifier(
    num_features=latent_size,
    hidden_dim=[32, 32],
    output_dim=1,
    activation=nn.LeakyReLU(0.2),
    batch_norm=True,
)
model_D = MLPClassifier(
    num_features=1,
    hidden_dim=[32, 32],
    output_dim=1,
    activation=nn.LeakyReLU(0.2),
    batch_norm=True,
)

In [ ]:
# Train
model_G, train_hist_G, train_hist_D, val_hist_G, val_hist_D = train_gan_model(
    model_G,
    model_D,
    train_loader,
    val_loader,
    latent_size,
    learning_rate,
    num_epochs,
    device,
    model_path="./outputs/vanilla_gan_G.pth",
    label_smoothing=0.1,
    g_use_sigmoid=False,  # Scaled age data has negative values
)

In [ ]:
# Plot training history
plt.figure(figsize=(10, 10))
plt.plot(train_hist_G, label="Train G Loss")
plt.plot(train_hist_D, label="Train D Loss")
plt.plot(val_hist_G, label="Val G Loss", linestyle="--")
plt.plot(val_hist_D, label="Val D Loss", linestyle="--")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend(frameon=False)
plt.savefig("./outputs/vanilla_gan_history.png")
plt.show()

In [ ]:
# Generate data and compare distributions
model_G.eval()
with torch.no_grad():
    noise = torch.randn(n_samples, latent_size).to(device)
    generated_age_scaled = model_G(noise, use_sigmoid=False).cpu().numpy()

generated_age = scaler.inverse_transform(generated_age_scaled)

In [ ]:
# Plot comparison with pull
n_bins = np.linspace(0, 110, 31)
plot_comparison_with_pull(
    age_values.flatten(),
    generated_age.flatten(),
    bins=n_bins,
    title="Age Distribution Comparison - Vanilla GAN",
    save_path="./outputs/vanilla_gan_comparison.png",
    label_gen="Generated (Vanilla GAN)",
)

## Instance Noise Method

In [ ]:
instance_noise_std = 0.1

In [ ]:
# Models
model_G = MLPClassifier(
    num_features=latent_size,
    hidden_dim=[32, 32],
    output_dim=1,
    activation=nn.LeakyReLU(0.2),
    batch_norm=True,
)
model_D = MLPClassifier(
    num_features=1,
    hidden_dim=[32, 32],
    output_dim=1,
    activation=nn.LeakyReLU(0.2),
    batch_norm=True,
)

In [ ]:
# Train
model_G, train_hist_G, train_hist_D, val_hist_G, val_hist_D = train_gan_model(
    model_G,
    model_D,
    train_loader,
    val_loader,
    latent_size,
    learning_rate,
    num_epochs,
    device,
    model_path="./outputs/instance_noise_gan_G.pth",
    instance_noise_std=instance_noise_std,
    label_smoothing=0.1,
    g_use_sigmoid=False,
)

In [ ]:
# Plot training history
plt.figure(figsize=(10, 10))
plt.plot(train_hist_G, label="Train G Loss")
plt.plot(train_hist_D, label="Train D Loss")
plt.plot(val_hist_G, label="Val G Loss", linestyle="--")
plt.plot(val_hist_D, label="Val D Loss", linestyle="--")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend(frameon=False)
plt.savefig("./outputs/instance_noise_gan_history.png")
plt.show()

In [ ]:
# Generate data and compare distributions
model_G.eval()
with torch.no_grad():
    noise = torch.randn(n_samples, latent_size).to(device)
    generated_age_scaled = model_G(noise, use_sigmoid=False).cpu().numpy()

generated_age = scaler.inverse_transform(generated_age_scaled)

In [ ]:
# Plot comparison with pull
n_bins = np.linspace(0, 110, 31)
plot_comparison_with_pull(
    age_values.flatten(),
    generated_age.flatten(),
    bins=n_bins,
    title="Age Distribution Comparison - Instance Noise GAN",
    save_path="./outputs/instance_noise_gan_comparison.png",
    label_gen="Generated (Instance Noise GAN)",
)

## DCTR Reweighting

In [ ]:
# Load pre-trained Vanilla GAN
model_G = MLPClassifier(
    num_features=latent_size,
    hidden_dim=[32, 32],
    output_dim=1,
    activation=nn.LeakyReLU(0.2),
    batch_norm=True,
)
model_G.load_state_dict(torch.load("./outputs/vanilla_gan_G.pth", map_location=device))
model_G.to(device)
model_G.eval()

In [ ]:
# Generate data from GAN
with torch.no_grad():
    noise = torch.randn(len(age_scaled), latent_size).to(device)
    # Use use_sigmoid=False as per Vanilla GAN training
    generated_age_scaled = model_G(noise, use_sigmoid=False).cpu().numpy()

In [ ]:
# Prepare data for DCTR classifier
# Target (real) = 1, Generated = 0
X = np.vstack([age_scaled, generated_age_scaled])
y = np.hstack([np.ones(len(age_scaled)), np.zeros(len(generated_age_scaled))])

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.5)

batch_size = 32
train_loader = DataLoader(
    TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train).float()),
    batch_size=batch_size,
    shuffle=True,
)
val_loader = DataLoader(
    TensorDataset(torch.from_numpy(X_val), torch.from_numpy(y_val).float()),
    batch_size=batch_size,
    shuffle=False,
)

In [ ]:
# DCTR Classifier
model_D_DCTR = MLPClassifier(
    num_features=1,
    hidden_dim=[32, 32],
    output_dim=1,
    activation=nn.ReLU(),
    dropout=0.0,
).to(device)

In [ ]:
optimizer = optim.Adam(model_D_DCTR.parameters(), lr=0.001)
criterion = nn.BCELoss()

In [ ]:
# Train DCTR Classifier
num_epochs = 50
for epoch in range(num_epochs):
    model_D_DCTR.train()
    for x_batch, y_batch in train_loader:
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        y_pred = model_D_DCTR(x_batch).view(-1)
        loss = criterion(y_pred, y_batch)
        loss.backward()
        optimizer.step()

    model_D_DCTR.eval()
    val_loss = 0
    with torch.no_grad():
        for x_batch, y_batch in val_loader:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            y_pred = model_D_DCTR(x_batch).view(-1)
            val_loss += criterion(y_pred, y_batch).item()
    print(f"DCTR Epoch {epoch + 1} Val Loss: {val_loss / len(val_loader):.4f}")

In [ ]:
# Save DCTR classifier
torch.save(model_D_DCTR.state_dict(), "./outputs/dctr_classifier.pth")

# Generate test data from GAN
with torch.no_grad():
    noise = torch.randn(n_samples, latent_size).to(device)
    # Use use_sigmoid=False as per Vanilla GAN training
    generated_age_scaled = model_G(noise, use_sigmoid=False).cpu().numpy()

In [ ]:
# Calculate weights for generated samples
model_D_DCTR.eval()
with torch.no_grad():
    y_pred_gen = (
        model_D_DCTR(torch.from_numpy(generated_age_scaled).to(device)).cpu().numpy()
    )
    # w = p_target / p_generated = D / (1 - D)
    weights = y_pred_gen / (1.0 - y_pred_gen + 1e-7)

In [ ]:
# Inverse transform generated data
generated_age = scaler.inverse_transform(generated_age_scaled)

# Plot comparison with pull
n_bins = np.linspace(0, 110, 31)
plot_comparison_with_pull(
    age_values.flatten(),
    generated_age.flatten(),
    bins=n_bins,
    title="Age Distribution Comparison - DCTR Reweighting",
    save_path="./outputs/dctr_reweighting_comparison.png",
    label_gen="Generated (DCTR Reweighted)",
    gen_weights=weights.flatten(),
)

## VAE model

In [ ]:
latent_dim = 16

In [ ]:
# Create DataLoaders
batch_size = 32
train_loader = DataLoader(
    TensorDataset(torch.from_numpy(train_age)), batch_size=batch_size, shuffle=True
)
val_loader = DataLoader(
    TensorDataset(torch.from_numpy(val_age)), batch_size=batch_size, shuffle=False
)

In [ ]:
model_VAE = MLPVAE(num_features=1, hidden_dim=[32, 32], latent_dim=latent_dim)

# Train
model_VAE, train_hist, val_hist = train_vae_model(
    model_VAE,
    train_loader,
    val_loader,
    learning_rate,
    num_epochs,
    device,
    model_path="./outputs/vae_model.pth",
)

In [ ]:
# Plot training history
plt.figure(figsize=(10, 10))
plt.plot(train_hist, label="Train Loss")
plt.plot(val_hist, label="Val Loss", linestyle="--")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend(frameon=False)
plt.savefig("./outputs/vae_history.png")
plt.show()

In [ ]:
# Generate data and compare distributions
model_VAE.eval()
with torch.no_grad():
    z = torch.randn(len(age_values), latent_dim).to(device)
    generated_age_scaled = model_VAE.decode(z).cpu().numpy()

In [ ]:
# Plot comparison with pull
n_bins = np.linspace(0, 110, 31)
plot_comparison_with_pull(
    age_values.flatten(),
    generated_age.flatten(),
    bins=n_bins,
    title="Age Distribution Comparison - VAE",
    save_path="./outputs/vae_comparison.png",
    label_gen="Generated (VAE)",
)